In [1]:
#  Load Libraries & Data
import pandas as pd
import numpy as np
matches    = pd.read_csv(r'C:\my project\ipl-analytics-command-center\data\matches_clean.csv')
deliveries = pd.read_csv(r'C:\my project\ipl-analytics-command-center\data\deliveries_clean.csv')
print("Matches    :", matches.shape)
print("Deliveries :", deliveries.shape)

Matches    : (1095, 19)
Deliveries : (260920, 20)


In [2]:
#  Merge Matches & Deliveries
df = deliveries.merge(matches[['Id','Season','Venue','Team1','Team2','Winner']], 
                      left_on='Match Id', right_on='Id').drop(columns='Id')

print(df.shape)
print(df.columns.tolist())

(260920, 25)
['Match Id', 'Inning', 'Batting Team', 'Bowling Team', 'Over', 'Ball', 'Batter', 'Bowler', 'Non Striker', 'Batsman Runs', 'Extra Runs', 'Total Runs', 'Extras Type', 'Is Wicket', 'Player Dismissed', 'Dismissal Kind', 'Fielder', 'Is Four', 'Is Six', 'Is Boundary', 'Season', 'Venue', 'Team1', 'Team2', 'Winner']


In [3]:
# MODEL 1 — WIN PROBABILITY PREDICTOR
#  Ball by Ball Features

df = df[df['Inning']==1].sort_values(['Match Id','Over','Ball'])
g  = df.groupby('Match Id')

df['Runs So Far'], df['Wickets So Far'] = g['Total Runs'].cumsum(), g['Is Wicket'].cumsum()
df['Balls So Far'] = g.cumcount() + 1
df['Balls Left']   = 120 - df['Balls So Far']
df['CRR']          = df['Runs So Far'] / df['Balls So Far'] * 6

print(df[['Over','Runs So Far','Wickets So Far','Balls Left','CRR']].head())

   Over  Runs So Far  Wickets So Far  Balls Left  CRR
0     1            1               0         119  6.0
1     1            1               0         118  3.0
2     1            2               0         117  4.0
3     1            2               0         116  3.0
4     1            2               0         115  2.4


In [4]:
#  Add Target & Winner Label

target = df.groupby('Match Id')['Total Runs'].sum().reset_index(name='Total Score')
df     = df.merge(target, on='Match Id')

df['Target']  = df['Total Score'] + 1
df['RRR']     = (df['Target'] - df['Runs So Far']) / (df['Balls Left'] / 6)
df['Win']     = (df['Batting Team'] == df['Winner']).astype(int)

print(df[['Over','Runs So Far','Target','RRR','Win']].head())

   Over  Runs So Far  Target        RRR  Win
0     1            1     223  11.193277    1
1     1            1     223  11.288136    1
2     1            2     223  11.333333    1
3     1            2     223  11.431034    1
4     1            2     223  11.530435    1


In [5]:
#  Prepare Features
features = ['Runs So Far','Balls So Far','Wickets So Far','Balls Left','CRR','RRR']
X = df[features]
y = df['Win']
print("X shape:", X.shape)
print("Class balance:\n", y.value_counts())

X shape: (135018, 6)
Class balance:
 Win
0    72956
1    62062
Name: count, dtype: int64


In [6]:
# Train Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (108014, 6)
Test : (27004, 6)


In [7]:
#  WIN PROBABILITY PREDICTOR
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# Step 1 — Fix inf BEFORE splitting
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

# Step 2 — Split AFTER cleaning
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 3 — Train
model_wp = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
model_wp.fit(X_train, y_train)

print(" Model Trained!")

 Model Trained!


In [8]:
#  WIN PROBABILITY PREDICTOR
from sklearn.metrics import accuracy_score, roc_auc_score
y_pred = model_wp.predict(X_test)
y_prob = model_wp.predict_proba(X_test)[:,1]
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}")

Accuracy : 0.6849
ROC-AUC  : 0.7581


In [9]:
#  WIN PROBABILITY PREDICTOR
import pickle, os
os.makedirs(r'C:\my project\ipl-analytics-command-center\ml_models', exist_ok=True)
with open(r'C:\my project\ipl-analytics-command-center\ml_models\win_probability_model.pkl', 'wb') as f:
    pickle.dump(model_wp, f)
print(" Win Probability Model Saved!")

 Win Probability Model Saved!


In [10]:
#  MODEL 2 — SCORE PREDICTOR
X2 = df[['Runs So Far','Balls So Far','Wickets So Far','Balls Left','CRR']].replace([np.inf,-np.inf],0).fillna(0)
y2 = df.groupby('Match Id')['Total Runs'].transform('sum')
print(X2.shape, y2.describe())

(135018, 5) count    135018.000000
mean        166.289221
std          31.534128
min          56.000000
25%         146.000000
50%         165.000000
75%         187.000000
max         287.000000
Name: Total Runs, dtype: float64


In [11]:
#  SCORE PREDICTOR
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)
model_sp = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42).fit(X2_train, y2_train)

y2_pred = model_sp.predict(X2_test)
print(f"MAE: {mean_absolute_error(y2_test, y2_pred):.2f} | R²: {r2_score(y2_test, y2_pred):.4f}")

MAE: 15.63 | R²: 0.5311


In [12]:
#  SCORE PREDICTOR
with open(r'C:\my project\ipl-analytics-command-center\ml_models\score_predictor_model.pkl', 'wb') as f:
    pickle.dump(model_sp, f)
print(" Score Predictor Saved!")

 Score Predictor Saved!


In [13]:
# MODEL 3 — Build Player Stats
bat = df.groupby('Batter').agg(
    Runs=('Batsman Runs','sum'),
    Balls=('Batsman Runs','count'),
    Sixes=('Is Six','sum'),
    Fours=('Is Four','sum')
).eval('SR = Runs / Balls * 100').reset_index()

bowl = df.groupby('Bowler').agg(
    Wickets=('Is Wicket','sum'),
    Runs_Given=('Total Runs','sum'),
    Balls_Bowled=('Total Runs','count')
).eval('Economy = Runs_Given / Balls_Bowled * 6').reset_index()

print(bat.shape, bowl.shape)

(596, 6) (480, 5)


In [14]:
# Score & Select Best XI
bat['Bat_Score']  = (bat['Runs']/bat['Runs'].max()*0.5) + (bat['SR']/bat['SR'].max()*0.3) + (bat['Sixes']/bat['Sixes'].max()*0.2)
bowl['Bowl_Score'] = (bowl['Wickets']/bowl['Wickets'].max()*0.6) + ((1 - bowl['Economy']/bowl['Economy'].max())*0.4)

top_bat  = bat.nlargest(7,'Bat_Score')[['Batter','Runs','SR','Bat_Score']]
top_bowl = bowl.nlargest(4,'Bowl_Score')[['Bowler','Wickets','Economy','Bowl_Score']]

print(" Top Batsmen:\n", top_bat.to_string(index=False))
print("\n Top Bowlers:\n", top_bowl.to_string(index=False))

 Top Batsmen:
         Batter  Runs         SR  Bat_Score
       V Kohli  4400 126.691621   0.747258
     Rg Sharma  3600 132.743363   0.662877
      Ch Gayle  2873 146.656457   0.636470
Ab De Villiers  3163 158.387581   0.636431
      S Dhawan  3926 123.575700   0.630360
     Da Warner  3280 133.932217   0.594569
      Ms Dhoni  3065 135.980479   0.594559

 Top Bowlers:
    Bowler  Wickets  Economy  Bowl_Score
Sp Narine      116 6.835634    0.851630
Pp Chawla      114 8.100674    0.813827
  B Kumar      110 7.470102    0.806824
 Ut Yadav      111 8.447484    0.790782


In [15]:
#  Save Best XI
import pandas as pd
best_xi = pd.concat([top_bat.rename(columns={'Batter':'Player'}), top_bowl.rename(columns={'Bowler':'Player'})], ignore_index=True)
best_xi.to_csv(r'C:\my project\ipl-analytics-command-center\ml_models\best_xi.csv', index=False)
print(" Best XI Saved!")

 Best XI Saved!


In [16]:
#  MODEL 4 — Player vs Opponent Features
p_stats = df.groupby(['Batter','Bowling Team']).agg(
    Runs=('Batsman Runs','sum'), Balls=('Batsman Runs','count'), Dismissals=('Is Wicket','sum')
).assign(SR=lambda x: x.Runs/x.Balls*100, Avg=lambda x: x.Runs/x.Dismissals.clip(1)).reset_index()
print(p_stats.shape, '\n', p_stats.head())

(2958, 7) 
            Batter           Bowling Team  Runs  Balls  Dismissals          SR  \
0  A Ashish Reddy    Chennai Super Kings    42     22           1  190.909091   
1  A Ashish Reddy  Kolkata Knight Riders    13      9           1  144.444444   
2  A Ashish Reddy         Mumbai Indians    27     25           2  108.000000   
3  A Ashish Reddy          Pune Warriors    26     19           0  136.842105   
4  A Ashish Reddy           Punjab Kings    22      8           0  275.000000   

    Avg  
0  42.0  
1  13.0  
2  13.5  
3  26.0  
4  22.0  


In [17]:
#  Train Player Performance Model
from sklearn.ensemble import RandomForestRegressor
X4 = p_stats[['Runs','Balls','Dismissals','SR']].fillna(0)
y4 = p_stats['Avg']
X4_train, X4_test, y4_train, y4_test = train_test_split(X4, y4, test_size=0.2, random_state=42)
model_pp = RandomForestRegressor(n_estimators=100, random_state=42).fit(X4_train, y4_train)
print(f"R²: {r2_score(y4_test, model_pp.predict(X4_test)):.4f}")

R²: 0.9928


In [18]:
#  Save Player Performance Model
with open(r'C:\my project\ipl-analytics-command-center\ml_models\player_performance_model.pkl', 'wb') as f:
    pickle.dump(model_pp, f)
print("Player Performance Model Saved!")

Player Performance Model Saved!


In [19]:
#  MODEL 5 — Toss Decision Recommender
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

X5 = pd.DataFrame({'Venue': LabelEncoder().fit_transform(matches['Venue']), 'Bat First Win': matches['Bat First Win']})
y5 = (matches['Toss Decision'] == 'Field').astype(int)

X5_train, X5_test, y5_train, y5_test = train_test_split(X5, y5, test_size=0.2, random_state=42)
model_td = LogisticRegression().fit(X5_train, y5_train)

print(f"Accuracy: {accuracy_score(y5_test, model_td.predict(X5_test)):.4f}")

Accuracy: 0.6575


In [20]:
# Save Toss Recommender
with open(r'C:\my project\ipl-analytics-command-center\ml_models\toss_recommender_model.pkl', 'wb') as f:
    pickle.dump(model_td, f)
print(" Toss Recommender Saved!")

 Toss Recommender Saved!


In [21]:
#  MODEL 6 — Auction Value Predictor (Fixed)
from sklearn.ensemble import GradientBoostingRegressor

auction = bat.merge(bowl, left_on='Batter', right_on='Bowler', how='outer').fillna(0)
auction.columns = auction.columns.str.replace('_x','_bat').str.replace('_y','_bowl')

auction['Price'] = (auction['Runs']/auction['Runs'].max()*10 +
                    auction['SR']/auction['SR'].max()*5  +
                    auction['Wickets']/auction['Wickets'].max()*8 +
                    (1 - auction['Economy']/auction['Economy'].max())*4)

X6 = auction[['Runs','SR','Fours','Sixes','Wickets','Economy']].fillna(0)
y6 = auction['Price']

X6_train, X6_test, y6_train, y6_test = train_test_split(X6, y6, test_size=0.2, random_state=42)
model_av = GradientBoostingRegressor(n_estimators=100, random_state=42).fit(X6_train, y6_train)

print(f"R²: {r2_score(y6_test, model_av.predict(X6_test)):.4f}")

R²: 0.9825


In [22]:
#  Save Auction Value Model
with open(r'C:\my project\ipl-analytics-command-center\ml_models\auction_value_model.pkl', 'wb') as f:
    pickle.dump(model_av, f)
print(" Auction Value Model Saved!")

 Auction Value Model Saved!


In [23]:
# Get venue encoding map
from sklearn.preprocessing import LabelEncoder
import pandas as pd

matches = pd.read_csv(r'C:\my project\ipl-analytics-command-center\data\matches_clean.csv')
le = LabelEncoder()
le.fit(matches['Venue'])
venue_map = {venue: int(code) for code, venue in enumerate(le.classes_)}
print(venue_map)

{'Arun Jaitley Stadium': 0, 'Arun Jaitley Stadium, Delhi': 1, 'Barabati Stadium': 2, 'Barsapara Cricket Stadium, Guwahati': 3, 'Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow': 4, 'Brabourne Stadium': 5, 'Brabourne Stadium, Mumbai': 6, 'Buffalo Park': 7, 'De Beers Diamond Oval': 8, 'Dr Dy Patil Sports Academy': 9, 'Dr Dy Patil Sports Academy, Mumbai': 10, 'Dr. Y.S. Rajasekhara Reddy Aca-Vdca Cricket Stadium': 11, 'Dr. Y.S. Rajasekhara Reddy Aca-Vdca Cricket Stadium, Visakhapatnam': 12, 'Dubai International Cricket Stadium': 13, 'Eden Gardens': 14, 'Eden Gardens, Kolkata': 15, 'Feroz Shah Kotla': 16, 'Green Park': 17, 'Himachal Pradesh Cricket Association Stadium': 18, 'Himachal Pradesh Cricket Association Stadium, Dharamsala': 19, 'Holkar Cricket Stadium': 20, 'Jsca International Stadium Complex': 21, 'Kingsmead': 22, 'M Chinnaswamy Stadium': 23, 'M Chinnaswamy Stadium, Bengaluru': 24, 'M.Chinnaswamy Stadium': 25, 'Ma Chidambaram Stadium': 26, 'Ma Chidambaram Sta

In [24]:
# Create Analytics Summary CSVs
path = r'C:\my project\ipl-analytics-command-center\webapp'

deliveries.groupby('Batter')['Batsman Runs'].sum().reset_index(name='Runs').sort_values('Runs',ascending=False).head(10).to_csv(f'{path}\\top_batsmen.csv',index=False)
matches.groupby('Season').size().reset_index(name='Matches').to_csv(f'{path}\\season_matches.csv',index=False)
matches[matches['Winner']!='No Result'].groupby('Winner').size().reset_index(name='Wins').sort_values('Wins',ascending=False).head(8).to_csv(f'{path}\\team_wins.csv',index=False)

print(" Done!")

 Done!
